# UC-DIM-2 — Administrative Hierarchy Navigation

**Persona:** FAO Regional Analyst

**Goal:** Navigate a 4-level administrative dimension (global → region → country →
admin-level-1), then retrieve the ancestor chain for a specific node to support
breadcrumb navigation in a spatial drill-down UI.

**Use case:** UC-DIM-2 — spatial scope selection for regional crop assessments

**Prerequisites:**
- `extension_dimensions` installed and registered
- Target collection has an `admin` dimension of type `hierarchical`
- Dimension populated with seed nodes: `AFR` (Africa region), `ETH` (Ethiopia),
  `ET_AM` (Amhara admin-level-1)
- Env vars: `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
TOKEN = os.environ["DYNASTORE_TOKEN"]
CATALOG_ID = os.environ["CATALOG_ID"]
COLLECTION_ID = os.environ.get("COLLECTION_ID", "fao_crop_monitoring")

DIM_BASE = f"/dimensions/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/dimensions"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)
print(f"BASE_URL     : {BASE_URL}")
print(f"CATALOG_ID   : {CATALOG_ID}")
print(f"COLLECTION_ID: {COLLECTION_ID}")

## Step 1 — Inspect the admin dimension

Fetch the `admin` dimension metadata. Confirm it is a hierarchical dimension and
inspect the reported level count. A 4-level hierarchy (global / region / country /
admin-level-1) should report `level_count: 4` or an equivalent `levels` array.

In [ ]:
resp = client.get(f"{DIM_BASE}/admin")
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

admin_dim = resp.json()
print(f"id         : {admin_dim.get('id')}")
print(f"type       : {admin_dim.get('type')}")
print(f"level_count: {admin_dim.get('level_count', admin_dim.get('levels', '—'))}")
print(f"description: {admin_dim.get('description', '—')}")

dim_type = admin_dim.get("type", "")
assert "hierarch" in dim_type.lower() or admin_dim.get("level_count") is not None, (
    f"Expected a hierarchical dimension type, got type={dim_type!r}"
)
print("Hierarchical dimension confirmed.")

## Step 2 — Navigate Africa children

Request the direct children of the `AFR` (Africa) region node. The response contains
country-level nodes, each identified by its ISO 3166-1 alpha-3 code. This is the call
a UI makes when the analyst expands a region in the spatial selector.

In [ ]:
resp = client.get(f"{DIM_BASE}/admin/children", params={"parent_id": "AFR"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
afr_children = data.get("members", data if isinstance(data, list) else [])

print(f"Africa child nodes ({len(afr_children)}):")
for node in afr_children:
    print(f"  id={node.get('id')}  label={node.get('label', node.get('title', '—'))}")

assert len(afr_children) > 0, "Africa (AFR) should have at least one child country node"
country_ids = [n.get("id") for n in afr_children]
print(f"\nCountry codes: {country_ids}")

## Step 3 — Navigate Ethiopia

Drill into Ethiopia (`ETH`) to retrieve its admin-level-1 regions. This is the third
level of the hierarchy. The response should include Amhara (`ET_AM`), Oromia
(`ET_OR`), SNNPR, and Tigray among others.

In [ ]:
resp = client.get(f"{DIM_BASE}/admin/children", params={"parent_id": "ETH"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
eth_children = data.get("members", data if isinstance(data, list) else [])

print(f"Ethiopia admin-level-1 regions ({len(eth_children)}):")
for node in eth_children:
    print(f"  id={node.get('id')}  label={node.get('label', node.get('title', '—'))}")

assert len(eth_children) > 0, "Ethiopia (ETH) must have at least one admin-level-1 child"
region_ids = [n.get("id") for n in eth_children]
assert "ET_AM" in region_ids, f"Expected ET_AM (Amhara) in Ethiopia children; got {region_ids}"
print(f"\nET_AM (Amhara) confirmed in Ethiopia's admin-level-1 regions.")

## Step 4 — Ancestor chain (breadcrumb)

Given the leaf node `ET_AM` (Amhara), retrieve its full ancestor chain to render a
breadcrumb: `Global > Africa > Ethiopia > Amhara`. The `/ancestors` endpoint returns
nodes ordered from root to parent, not including the node itself.

In [ ]:
resp = client.get(f"{DIM_BASE}/admin/ancestors", params={"node_id": "ET_AM"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
ancestors = data.get("members", data if isinstance(data, list) else [])

ancestor_ids = [n.get("id") for n in ancestors]
print(f"Ancestor chain for ET_AM: {ancestor_ids}")

assert "ETH" in ancestor_ids, (
    f"Expected ETH (Ethiopia) in ancestor chain of ET_AM; got {ancestor_ids}"
)
assert "AFR" in ancestor_ids, (
    f"Expected AFR (Africa) in ancestor chain of ET_AM; got {ancestor_ids}"
)
print("Ancestor chain verified: AFR and ETH both present.")

# Order: AFR must appear before ETH (root-first ordering)
afr_pos = ancestor_ids.index("AFR")
eth_pos = ancestor_ids.index("ETH")
assert afr_pos < eth_pos, (
    f"Expected root-first ordering: AFR ({afr_pos}) before ETH ({eth_pos})"
)
print("Root-first ordering confirmed.")

## Edge cases

### Unknown parent_id → 404

Requesting children of a node that does not exist must return 404, not an empty list.
This allows clients to distinguish "no children" from "node not found".

In [ ]:
resp = client.get(f"{DIM_BASE}/admin/children", params={"parent_id": "DOES_NOT_EXIST"})
print(resp.status_code, resp.text[:200])
assert resp.status_code == 404, (
    f"Expected 404 for unknown parent_id, got {resp.status_code}: {resp.text}"
)
print("404 confirmed for unknown parent_id — correct behaviour.")

In [ ]:
# Leaf node children → 200 with empty list, not 404
# ET_AM is a leaf (no further subdivisions in this dataset)
resp = client.get(f"{DIM_BASE}/admin/children", params={"parent_id": "ET_AM"})
print(resp.status_code)
assert resp.status_code == 200, (
    f"Expected 200 (empty list) for leaf node ET_AM, got {resp.status_code}: {resp.text}"
)

data = resp.json()
leaf_children = data.get("members", data if isinstance(data, list) else [])
assert len(leaf_children) == 0, (
    f"Expected 0 children for leaf node ET_AM, got {len(leaf_children)}"
)
print("Empty list (not 404) confirmed for leaf node ET_AM — correct behaviour.")

In [ ]:
# Recursive vs leveled distinction
# The response schema must include either `depth` (recursive) or `level` (leveled).
# Both are valid dimension implementations; this cell documents which mode is active.
resp = client.get(f"{DIM_BASE}/admin/children", params={"parent_id": "AFR"})
assert resp.status_code == 200

data = resp.json()
sample_nodes = data.get("members", data if isinstance(data, list) else [])

if sample_nodes:
    first = sample_nodes[0]
    has_depth = "depth" in first
    has_level = "level" in first
    print(f"Sample node keys: {list(first.keys())}")
    print(f"depth field present: {has_depth}")
    print(f"level field present: {has_level}")
    assert has_depth or has_level, (
        f"Expected either 'depth' or 'level' field in node; got keys: {list(first.keys())}"
    )
    mode = "recursive (depth)" if has_depth else "leveled (level)"
    print(f"Hierarchy mode: {mode}")
else:
    print("No nodes to inspect schema from — skipping depth/level assertion.")

client.close()